# psfFlux / scienceFlux / templateFlux — Flux curves + CCD trajectory on focal plane

## Purpose

This notebook investigates the origin of light-curve variability for
stable stars by correlating flux variations with the **trajectory of
observations across the focal plane** (i.e., which CCD was hit at each
epoch).

### Layout per DIA object — **2 rows × 6 columns** (one column per band `u g r i z y`)

```
  ┌────────────────────────────────────────────────────────────────┐
  │  Row 0 (top) :  psfFlux / scienceFlux / templateFlux  vs Δt   │
  │               colour = Δt (cmap jet: blue=0, red=max)         │
  ├────────────────────────────────────────────────────────────────┤
  │  Row 1 (bot) :  LSSTCam focal plane — CCD patches (grey)      │
  │               annotated detector numbers only                  │
  │               coloured scatter of visited CCDs vs time         │
  │               (same jet cmap as row 0 → time trajectory)       │
  └────────────────────────────────────────────────────────────────┘
```

**Key diagnostic:** if flux anomalies in row 0 cluster on the same CCD
in row 1, that CCD is likely responsible for the observed variability.

When a CCD is visited multiple times, each visit is plotted as a
separate scatter point with a small random jitter so that all points
remain visible.

---

- **Author:** Sylvie Dagoret-Campagne  
- **Affiliation:** IJCLab / IN2P3 / CNRS  
- **Follows:** `07c_psfFlux_scienceFlux_templateFlux_showdetector.ipynb`
- **Creation date:** 2026-04-07  
- **Last update:** 2026-04-12  
- **Subject:** Fink alert broker — Rubin LSST DIA photometry diagnostics vs focal-plane trajectory

## 1. Imports & configuration

In [ ]:
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as mcm
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
from astropy.time import Time

warnings.filterwarnings("ignore")

print(f"pandas  version : {pd.__version__}")
print(f"numpy   version : {np.__version__}")

In [ ]:
try:
    import ipympl  # noqa: F401

    %matplotlib widget
    print("ipympl found → interactive backend (%matplotlib widget)")
except ImportError:
    %matplotlib inline
    print("ipympl NOT found → falling back to %matplotlib inline")

In [ ]:
# ── Notebook tag & paths ──────────────────────────────────────────────────────
NB_TAG = "FINK_BLOCK_LC_07e"
DIR_DATA_IN = "data_FINK_BLOCK_LC_03"
DIR_FIGS = f"figs_{NB_TAG}"
os.makedirs(DIR_FIGS, exist_ok=True)

FILE_BUTLER = os.path.join(DIR_DATA_IN, "src_joined_butler.parquet")
FILE_CONSDB = os.path.join(DIR_DATA_IN, "src_joined_consdb.parquet")
GEOMETRY_CSV = "data_FocalPlane/ccd_geometry.csv"  # same as notebooks 05..

N_TOP = 20  # top-N objects by alert count

BANDS = list("ugrizy")  # 6 bands — no combined panel in this notebook
N_BANDS = len(BANDS)  # = 6

BAND_COLORS = {
    "u": "#9b59b6",
    "g": "#2ecc71",
    "r": "#e74c3c",
    "i": "#e67e22",
    "z": "#3498db",
    "y": "#795548",
}

# ── Column names ──────────────────────────────────────────────────────────────
COL_OBJ = "r:diaObjectId"
COL_SRC = "r:diaSourceId"
COL_MJD = "r:midpointMjdTai"
COL_BAND = "r:band"
COL_PSF = "r:psfFlux"
COL_PSFERR = "r:psfFluxErr"
COL_DET = "r:detector"  # CCD number — must be present in Butler-joined parquet

SCIENCE_FLUX_CANDIDATES = [
    "r:scienceFlux",
    "r:psfScience",
    "r:psfScienceFlux",
    "scienceFlux",
    "psfScience",
    "psfScienceFlux",
]
SCIENCE_ERR_CANDIDATES = [
    "r:scienceSigma",
    "r:scienceFluxErr",
    "r:psfScienceSigma",
    "r:psfScienceFluxErr",
    "scienceSigma",
    "scienceFluxErr",
    "psfScienceSigma",
]
TEMPLATE_FLUX_CANDIDATES = [
    "psfTemplateFlux",
    "r:templateFlux",
    "r:template",
    "templateFlux",
    "template",
]
TEMPLATE_ERR_CANDIDATES = [
    "r:templaeSigma",
    "r:templateFluxErr",
    "r:psfTemplateSigma",
    "r:psfTemplateFluxErr",
    "templateSigma",
    "templateFluxErr",
    "psfTemplateSigma",
]

# ── Colourmap for time axis (both flux curves and focal-plane trajectory) ─────
CMAP_TIME = mcm.get_cmap("jet")

plt.rcParams.update(
    {
        "figure.dpi": 120,
        "axes.grid": True,
        "grid.alpha": 0.3,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "font.size": 9,
    }
)


def savefig(name):
    """Save current figure as both PDF and PNG in DIR_FIGS."""
    for ext in ("pdf", "png"):
        plt.savefig(os.path.join(DIR_FIGS, f"{name}.{ext}"), bbox_inches="tight")
    print(f"  → saved {name}.{{pdf,png}}")


print(f"Input directory  : {os.path.abspath(DIR_DATA_IN)}")
print(f"Figure directory : {os.path.abspath(DIR_FIGS)}")
print(f"Geometry CSV     : {os.path.abspath(GEOMETRY_CSV)}")
print(f"N_TOP            : {N_TOP}")

## 2. Load alert data

In [ ]:
if os.path.exists(FILE_BUTLER):
    df = pd.read_parquet(FILE_BUTLER)
    src_label = "butler"
    print(f"Loaded butler file : {FILE_BUTLER}")
elif os.path.exists(FILE_CONSDB):
    df = pd.read_parquet(FILE_CONSDB)
    src_label = "consdb"
    print(f"Butler file not found. Loaded consDb file : {FILE_CONSDB}")
else:
    raise FileNotFoundError(
        f"Neither {FILE_BUTLER} nor {FILE_CONSDB} found.\n"
        "Run notebook 03_associateFinkAlerts-RubinVisits.ipynb first."
    )

print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns  |  source: {src_label}")

## 3. Schema inspection — detect flux columns & detector column

In [ ]:
print("All columns:")
for col in sorted(df.columns):
    print(f"  {col:50s}  dtype={df[col].dtype}")

In [ ]:
# ── scienceFlux ───────────────────────────────────────────────────────────────
COL_SCI, COL_SCIERR = None, None
for c in SCIENCE_FLUX_CANDIDATES:
    if c in df.columns:
        COL_SCI = c
        break
for c in SCIENCE_ERR_CANDIDATES:
    if c in df.columns:
        COL_SCIERR = c
        break

# ── templateFlux ──────────────────────────────────────────────────────────────
COL_TEMP, COL_TEMPERR = None, None
for c in TEMPLATE_FLUX_CANDIDATES:
    if c in df.columns:
        COL_TEMP = c
        break
for c in TEMPLATE_ERR_CANDIDATES:
    if c in df.columns:
        COL_TEMPERR = c
        break

has_science = COL_SCI is not None
has_sci_err = COL_SCIERR is not None
has_template = COL_TEMP is not None
has_temp_err = COL_TEMPERR is not None

print(f"psfFlux      : {COL_PSF}")
print(f"scienceFlux  : {COL_SCI}")
print(f"scienceErr   : {COL_SCIERR}")
print(f"templateFlux : {COL_TEMP}")
print(f"templateErr  : {COL_TEMPERR}")

# ── Detector column ───────────────────────────────────────────────────────────
if COL_DET not in df.columns:
    raise KeyError(
        f"Detector column '{COL_DET}' not found.\n"
        "This notebook requires Butler-joined data (src_joined_butler.parquet)."
    )
print(f"\nDetector column: '{COL_DET}'  — {df[COL_DET].nunique()} unique CCDs")

## 4. Load focal-plane geometry

In [ ]:
geom = pd.read_csv(GEOMETRY_CSV)
print(f"Geometry: {len(geom)} CCDs  |  columns: {list(geom.columns)}")

# Focal-plane bounding box (used for every subplot in row 1)
FP_XMIN = geom[[f"corner{i}_x" for i in range(4)]].min().min()
FP_XMAX = geom[[f"corner{i}_x" for i in range(4)]].max().max()
FP_YMIN = geom[[f"corner{i}_y" for i in range(4)]].min().min()
FP_YMAX = geom[[f"corner{i}_y" for i in range(4)]].max().max()

# Build quick lookup: detector_id → (x_center, y_center)
DET_POS = dict(zip(geom["detector"].astype(int), zip(geom["x_center"], geom["y_center"])))

print(f"Focal-plane extent: X=[{FP_XMIN:.0f}, {FP_XMAX:.0f}]  Y=[{FP_YMIN:.0f}, {FP_YMAX:.0f}] mm")

## 5. Rank DIA objects by alert count

In [ ]:
alert_counts = df.groupby(COL_OBJ).size().rename("n_alerts").sort_values(ascending=False)
print(f"Total unique DIA objects : {len(alert_counts):,}")
print(f"Top {N_TOP}:")
print(alert_counts.head(N_TOP).to_string())

In [ ]:
top_objects = alert_counts.head(N_TOP).index.tolist()
df_top = df[df[COL_OBJ].isin(top_objects)].copy()
df_top[COL_MJD] = df_top[COL_MJD].astype(float)
print(f"Rows kept: {len(df_top):,}")

## 6. Utility functions

In [ ]:
AB_FLUX_ZERO = 3631e9  # nJy at AB zero-point


def flux_to_mag(flux_nJy, flux_err_nJy=None):
    flux = np.asarray(flux_nJy, dtype=float)
    with np.errstate(invalid="ignore", divide="ignore"):
        mag = np.where(flux > 0, -2.5 * np.log10(flux / AB_FLUX_ZERO), np.nan)
    mag_err = None
    if flux_err_nJy is not None:
        err = np.asarray(flux_err_nJy, dtype=float)
        with np.errstate(invalid="ignore", divide="ignore"):
            mag_err = np.where(flux > 0, 2.5 / np.log(10) * np.abs(err / flux), np.nan)
    return mag, mag_err


def mjd_to_datestr(mjd_array):
    """Convert MJD array to ISO date strings 'YYYY-MM-DD'."""
    t = Time(np.asarray(mjd_array, dtype=float), format="mjd", scale="tai")
    return [tt.strftime("%Y-%m-%d") for tt in t]


def _add_date_axis(ax, dt_array, t0_mjd):
    """Secondary x-axis on top of *ax* showing calendar dates."""
    dt_finite = dt_array[np.isfinite(dt_array)]
    if len(dt_finite) == 0:
        return
    dt_min, dt_max = float(dt_finite.min()), float(dt_finite.max())
    if dt_max <= dt_min:
        tick_dt = np.array([dt_min])
    else:
        n_ticks = min(5, max(2, int((dt_max - dt_min) / 10) + 2))
        tick_dt = np.linspace(dt_min, dt_max, n_ticks)
    tick_dates = mjd_to_datestr(t0_mjd + tick_dt)
    ax2 = ax.twiny()
    ax2.set_xlim(ax.get_xlim())
    ax2.set_xticks(tick_dt)
    ax2.set_xticklabels(tick_dates, rotation=35, ha="left", fontsize=7)
    ax2.tick_params(axis="x", length=3)


def _shared_lim(arrays, margin=0.05):
    """Common [vmin, vmax] across a list of arrays, with margin."""
    all_vals = np.concatenate([np.asarray(a, dtype=float).ravel() for a in arrays])
    finite = all_vals[np.isfinite(all_vals)]
    if len(finite) == 0:
        return None
    vmin, vmax = finite.min(), finite.max()
    span = max(vmax - vmin, 1e-6)
    return vmin - margin * span, vmax + margin * span


def _dt_norm(dt_array, global_dt_max):
    """Normalise Δt values to [0, 1] using the global maximum Δt."""
    if global_dt_max <= 0:
        return np.zeros_like(dt_array, dtype=float)
    return np.clip(np.asarray(dt_array, dtype=float) / global_dt_max, 0, 1)


print("Utility functions defined.")

## 6b. Per-object pixel/flux table — `get_diaobject_table()`

This function extracts a **flat, analysis-ready DataFrame** for a single `diaObjectId`
from the already-loaded DataFrame `df` (the parquet file from Section 2).

It uses the column names resolved in Section 3 (`COL_SCI`, `COL_TEMP`, etc.) and
returns a clean table with standardised short column names.

| column | source column | description |
|---|---|---|
| `mjd` | `r:midpointMjdTai` | Midpoint of exposure (TAI, MJD) |
| `band` | `r:band` | Photometric band (`u/g/r/i/z/y`) |
| `visit` | `r:visit` | Butler visitId (Int64, nullable) |
| `detector` | `r:detector` | CCD detector number (Int64, nullable) |
| `x` | `r:x` | Centroid x pixel coordinate on CCD |
| `y` | `r:y` | Centroid y pixel coordinate on CCD |
| `psfFlux` | `r:psfFlux` | PSF flux of the difference-image source (nJy) |
| `psfFluxErr` | `r:psfFluxErr` | Uncertainty on psfFlux (nJy) |
| `scienceFlux` | `COL_SCI` | Flux on the science image (nJy); NaN if unavailable |
| `scienceFluxErr` | `COL_SCIERR` | Uncertainty on scienceFlux (nJy); NaN if unavailable |
| `templateFlux` | `COL_TEMP` | Flux on the template image (nJy); NaN if unavailable |
| `templateFluxErr` | `COL_TEMPERR` | Uncertainty on templateFlux (nJy); NaN if unavailable |

In [ ]:
def get_diaobject_table(diaObjectId, source_df=None) -> pd.DataFrame:
    """
    Return a flat analysis-ready DataFrame for a single diaObjectId.

    Parameters
    ----------
    diaObjectId : int or str
    source_df   : pd.DataFrame or None — defaults to module-level `df`.

    Returns
    -------
    pd.DataFrame with columns:
        diaObj, diaSrc, mjd, band, visit (Int64), detector (Int64), x, y,
        psfFlux, psfFluxErr, scienceFlux, scienceFluxErr,
        templateFlux, templateFluxErr, day_obs, month_obs
    """
    _src = source_df if source_df is not None else df
    mask = _src[COL_OBJ].astype(str) == str(diaObjectId)
    df_obj = _src[mask].copy()

    empty_cols = [
        "diaObj",
        "diaSrc",
        "mjd",
        "band",
        "visit",
        "detector",
        "x",
        "y",
        "psfFlux",
        "psfFluxErr",
        "scienceFlux",
        "scienceFluxErr",
        "templateFlux",
        "templateFluxErr",
        "day_obs",
        "month_obs",
    ]
    if df_obj.empty:
        return pd.DataFrame(columns=empty_cols)

    out = pd.DataFrame()
    out["diaObj"] = df_obj[COL_OBJ].values
    out["diaSrc"] = df_obj[COL_SRC].values
    out["mjd"] = pd.to_numeric(df_obj[COL_MJD], errors="coerce")
    out["band"] = df_obj[COL_BAND].values
    out["psfFlux"] = pd.to_numeric(df_obj[COL_PSF], errors="coerce")
    out["psfFluxErr"] = pd.to_numeric(df_obj[COL_PSFERR], errors="coerce")

    for col_in, col_out in (("r:visit", "visit"), ("r:detector", "detector")):
        if col_in in df_obj.columns:
            out[col_out] = pd.to_numeric(df_obj[col_in], errors="coerce").astype("Int64")
        else:
            out[col_out] = pd.array([pd.NA] * len(df_obj), dtype="Int64")

    for col_in, col_out in (("r:x", "x"), ("r:y", "y")):
        if col_in in df_obj.columns:
            out[col_out] = pd.to_numeric(df_obj[col_in], errors="coerce")
        else:
            out[col_out] = np.nan

    out["scienceFlux"] = (
        pd.to_numeric(df_obj[COL_SCI], errors="coerce")
        if COL_SCI is not None and COL_SCI in df_obj.columns
        else np.nan
    )
    out["scienceFluxErr"] = (
        pd.to_numeric(df_obj[COL_SCIERR], errors="coerce")
        if COL_SCIERR is not None and COL_SCIERR in df_obj.columns
        else np.nan
    )
    out["templateFlux"] = (
        pd.to_numeric(df_obj[COL_TEMP], errors="coerce")
        if COL_TEMP is not None and COL_TEMP in df_obj.columns
        else np.nan
    )
    out["templateFluxErr"] = (
        pd.to_numeric(df_obj[COL_TEMPERR], errors="coerce")
        if COL_TEMPERR is not None and COL_TEMPERR in df_obj.columns
        else np.nan
    )

    out = (
        out[
            [
                "diaObj",
                "diaSrc",
                "mjd",
                "band",
                "visit",
                "detector",
                "x",
                "y",
                "psfFlux",
                "psfFluxErr",
                "scienceFlux",
                "scienceFluxErr",
                "templateFlux",
                "templateFluxErr",
            ]
        ]
        .sort_values(["mjd", "band"], na_position="last")
        .reset_index(drop=True)
    )
    out["day_obs"] = out["visit"] // 100000
    out["month_obs"] = out["visit"] // 10000000
    return out


print("get_diaobject_table() defined.")

### 6b-demo — Quick test of `get_diaobject_table()`

In [ ]:
_test_oid = top_objects[1]
df_test = get_diaobject_table(_test_oid)
print(f"diaObjectId {_test_oid}  →  {len(df_test)} rows")
print(f"  bands present   : {sorted(df_test['band'].dropna().unique())}")
print(f"  visit range     : {df_test['visit'].dropna().min()} … {df_test['visit'].dropna().max()}")
print(f"  mjd  range      : {df_test['mjd'].min():.2f} … {df_test['mjd'].max():.2f}")
print(f"  scienceFlux NaN : {df_test['scienceFlux'].isna().sum()} / {len(df_test)}")
print(f"  templateFlux NaN: {df_test['templateFlux'].isna().sum()} / {len(df_test)}")
display(df_test)

## 6c. Load diaObject catalog from file

In [ ]:
DIR_CAT = "data_FINK_BLOCK_LC_01"
_cat_priority = [
    os.path.join(DIR_CAT, "diaobj_catalogue_gaia_stable.csv"),
    os.path.join(DIR_CAT, "diaobj_catalogue.csv"),
]
df_cat_ref = None
for _cat_path in _cat_priority:
    if os.path.exists(_cat_path):
        df_cat_ref = pd.read_csv(_cat_path, low_memory=False)
        df_cat_ref["diaObjectId"] = df_cat_ref["diaObjectId"].astype(str)
        print(f"Loaded catalogue : {_cat_path}  ({len(df_cat_ref)} objects)")
        break
if df_cat_ref is None:
    print("WARNING: diaObject catalogue not found — reference flux lines will be skipped.")

## 7. Focal-plane background drawing helper

Draw all CCD patches in grey with detector number annotations.
Reused for every per-band focal-plane subplot in row 1.

In [ ]:
def draw_focal_plane_background(ax):
    """
    Draw all LSSTCam CCD polygons as light-grey patches with detector-number
    annotations.  Axes limits and aspect are set to match the focal-plane
    bounding box.
    """
    for _, row_g in geom.iterrows():
        corners = [(row_g[f"corner{j}_x"], row_g[f"corner{j}_y"]) for j in range(4)]
        poly = mpatches.Polygon(
            corners,
            facecolor="lightgrey",
            edgecolor="black",
            linewidth=0.3,
            zorder=1,
        )
        ax.add_patch(poly)
        ax.text(
            row_g["x_center"],
            row_g["y_center"],
            str(int(row_g["detector"])),
            ha="center",
            va="center",
            fontsize=4,
            fontweight="bold",
            color="dimgrey",
            zorder=2,
        )
    ax.set_aspect("equal")
    ax.set_xlim(FP_XMIN, FP_XMAX)
    ax.set_ylim(FP_YMIN, FP_YMAX)
    ax.set_xticks([])
    ax.set_yticks([])


print("draw_focal_plane_background() defined.")

## 8. Main plot function — 2 rows × 6 columns per DIA object

**Row 0** — flux vs Δt, colour-coded by Δt (cmap `jet`: blue=0, red=max).  
**Row 1** — focal-plane CCD trajectory, same time colour coding.

When the same CCD is visited several times, points are jittered within the
CCD boundary to avoid complete overlap.

In [ ]:
# Typical CCD half-size in focal-plane coordinates (mm)
JITTER_FRAC = 0.30  # fraction of CCD half-size used as jitter amplitude


def _ccd_halfsize():
    """Estimate the typical CCD half-size in mm from the geometry CSV."""
    dx = (geom["corner1_x"] - geom["corner0_x"]).abs()
    dy = (geom["corner3_y"] - geom["corner0_y"]).abs()
    return float(np.nanmedian(np.concatenate([dx.values, dy.values]))) / 2.0


CCD_HALF = _ccd_halfsize()
JITTER_AMP = JITTER_FRAC * CCD_HALF
print(f"Estimated CCD half-size : {CCD_HALF:.1f} mm  →  jitter amplitude : {JITTER_AMP:.1f} mm")


def plot_flux_and_focalplane(obj_id, df_obj, src_label, rank):
    """
    Produce a 2×6 figure for one DIA object:
      - Row 0: psfFlux / scienceFlux / templateFlux vs Δt  (colour = Δt, jet)
      - Row 1: focal-plane CCD trajectory                  (same colour = Δt)

    Parameters
    ----------
    obj_id    : int/str  — diaObjectId
    df_obj    : DataFrame — rows for this object, sorted by MJD
    src_label : str      — 'butler' or 'consdb'
    rank      : int      — 0-based rank
    """

    # special with object catalog for flux straight-lines
    oid_str = str(obj_id)
    ref_psf = {}
    ref_sci = {}
    if df_cat_ref is not None:
        mask_cat = df_cat_ref["diaObjectId"].astype(str) == oid_str
        if mask_cat.any():
            cat_row = df_cat_ref[mask_cat].iloc[0]
            for b in BANDS:
                for dest, col in [(ref_psf, f"r:{b}_psfFluxMean"), (ref_sci, f"r:{b}_scienceFluxMean")]:
                    try:
                        fval = float(cat_row[col])
                        dest[b] = fval if np.isfinite(fval) else None
                    except (KeyError, TypeError, ValueError):
                        dest[b] = None

    rng = np.random.default_rng(seed=int(obj_id) % (2**31) if str(obj_id).isdigit() else 0)

    t0_mjd = float(df_obj[COL_MJD].min())
    t0_date = mjd_to_datestr([t0_mjd])[0]
    field = df_obj["field"].iloc[0] if "field" in df_obj.columns else ""

    global_dt_max = float((df_obj[COL_MJD].values.astype(float) - t0_mjd).max())
    if global_dt_max <= 0:
        global_dt_max = 1.0

    # ── Pre-compute per-band data ─────────────────────────────────────────────
    band_data = {}
    all_flux = []
    all_dt = []

    for band in BANDS:
        mask = df_obj[COL_BAND] == band
        df_b = df_obj[mask].sort_values(COL_MJD)
        if len(df_b) == 0:
            band_data[band] = None
            continue
        dt = df_b[COL_MJD].values.astype(float) - t0_mjd
        det_ids = df_b[COL_DET].values.astype(int)
        psf = df_b[COL_PSF].values.astype(float)
        psferr = df_b[COL_PSFERR].values.astype(float)
        sci = df_b[COL_SCI].values.astype(float) if has_science else None
        scierr = df_b[COL_SCIERR].values.astype(float) if has_sci_err else None
        temp = df_b[COL_TEMP].values.astype(float) if has_template else None
        temperr = df_b[COL_TEMPERR].values.astype(float) if has_temp_err else None

        dt_norm = _dt_norm(dt, global_dt_max)
        colors = CMAP_TIME(dt_norm)

        band_data[band] = dict(
            dt=dt,
            dt_norm=dt_norm,
            colors=colors,
            det_ids=det_ids,
            psf=psf,
            psferr=psferr,
            sci=sci,
            scierr=scierr,
            temp=temp,
            temperr=temperr,
        )
        all_dt.append(dt)
        all_flux.append(psf)
        if sci is not None:
            all_flux.append(sci)
        if temp is not None:
            all_flux.append(temp)

    ylim = _shared_lim(all_flux) if all_flux else None
    xlim = _shared_lim(all_dt) if all_dt else None

    # ── Figure layout: 2 rows × 6 cols ───────────────────────────────────────
    fig = plt.figure(figsize=(3.8 * N_BANDS, 9.0))
    gs = fig.add_gridspec(2, N_BANDS, height_ratios=[1.0, 1.0], hspace=0.45, wspace=0.40)
    axes_flux = [fig.add_subplot(gs[0, k]) for k in range(N_BANDS)]
    axes_fp = [fig.add_subplot(gs[1, k]) for k in range(N_BANDS)]

    # ── Row 0: flux panels ────────────────────────────────────────────────────
    for idx, band in enumerate(BANDS):
        ax = axes_flux[idx]
        bcolor = BAND_COLORS.get(band, "k")

        if band_data[band] is None:
            ax.text(
                0.5,
                0.5,
                "no data",
                ha="center",
                va="center",
                transform=ax.transAxes,
                color="gray",
                fontsize=8,
            )
            ax.set_title(f"band {band}", fontsize=9, color=bcolor)
            if xlim:
                ax.set_xlim(xlim)
            if ylim:
                ax.set_ylim(ylim)
            ax.set_xlabel("Δt [days]", fontsize=8)
            continue

        d = band_data[band]
        n_pts = len(d["dt"])

        if d["sci"] is not None:
            for i in range(n_pts):
                ax.errorbar(
                    d["dt"][i],
                    d["sci"][i],
                    yerr=d["scierr"][i] if d["scierr"] is not None else 0,
                    fmt="o",
                    ms=5,
                    color=d["colors"][i],
                    ecolor=d["colors"][i],
                    elinewidth=0.9,
                    capsize=2,
                    alpha=0.90,
                )

        for i in range(n_pts):
            ax.errorbar(
                d["dt"][i],
                d["psf"][i],
                yerr=d["psferr"][i],
                fmt="s",
                ms=4,
                color="none",
                mec=d["colors"][i],
                mew=1.1,
                ecolor=d["colors"][i],
                elinewidth=0.7,
                capsize=2,
                alpha=0.70,
            )

        if d["temp"] is not None:
            for i in range(n_pts):
                ax.errorbar(
                    d["dt"][i],
                    d["temp"][i],
                    yerr=d["temperr"][i] if d["temperr"] is not None else 0,
                    fmt="D",
                    ms=4,
                    color="none",
                    mec=d["colors"][i],
                    mew=1.0,
                    ecolor=d["colors"][i],
                    elinewidth=0.7,
                    capsize=2,
                    alpha=0.55,
                )

        # add object flux per band
        v_sci = ref_sci.get(band)
        if v_sci is not None:
            mag, merr = flux_to_mag(v_sci)
            ax.axhline(v_sci, color=bcolor, lw=1.2, ls="-.", alpha=0.85, label=f"obj: m={mag:.1f} mag")
        v_psf = ref_psf.get(band)
        if v_psf is not None:
            mag, merr = flux_to_mag(v_psf)
            ax.axhline(v_psf, color=bcolor, lw=1.2, ls=":", alpha=0.85, label=f"diaobj: m={mag:.1f} mag")

        ax.axhline(0.0, color="k", lw=0.6, ls=":", alpha=0.4)
        ax.set_title(f"band {band}\nsci● psf□ tmpl◇  (colour=Δt)", fontsize=8, color=bcolor)
        ax.set_ylabel("flux [nJy]", fontsize=8)
        ax.set_xlabel("Δt [days]", fontsize=8)
        ax.legend(fontsize=7, loc="best")

        if xlim:
            ax.set_xlim(xlim)
        if ylim:
            ax.set_ylim(ylim)
        _add_date_axis(ax, d["dt"], t0_mjd)

    # ── Row 1: focal-plane trajectory panels ──────────────────────────────────
    for idx, band in enumerate(BANDS):
        ax_fp = axes_fp[idx]
        bcolor = BAND_COLORS.get(band, "k")
        draw_focal_plane_background(ax_fp)

        if band_data[band] is None:
            ax_fp.set_title(f"band {band} — no data", fontsize=8, color=bcolor)
            continue

        d = band_data[band]
        n_pts = len(d["dt"])

        for i in range(n_pts):
            det_id = int(d["det_ids"][i])
            if det_id not in DET_POS:
                continue
            xc, yc = DET_POS[det_id]
            jx = rng.uniform(-JITTER_AMP, JITTER_AMP)
            jy = rng.uniform(-JITTER_AMP, JITTER_AMP)
            ax_fp.scatter(
                xc + jx, yc + jy, c=[d["colors"][i]], s=25, edgecolors="k", linewidths=0.4, zorder=5
            )

        ax_fp.set_title(f"band {band}  — CCD trajectory\n(colour = Δt, blue→red)", fontsize=8, color=bcolor)

    # ── Shared colour bar for the time axis ───────────────────────────────────
    sm = plt.cm.ScalarMappable(
        cmap=CMAP_TIME,
        norm=mcolors.Normalize(vmin=0, vmax=global_dt_max),
    )
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=axes_fp, fraction=0.018, pad=0.02)
    cbar.set_label("Δt [days] from first alert", fontsize=9)

    # ── Legend strip for marker types ────────────────────────────────────────
    type_handles = [
        Line2D(
            [0], [0], marker="o", color="grey", markersize=6, linewidth=0, label="● scienceFlux  (filled)"
        ),
        Line2D(
            [0],
            [0],
            marker="s",
            color="none",
            markeredgecolor="grey",
            markersize=6,
            linewidth=0,
            label="□ psfFlux      (open)",
        ),
        Line2D(
            [0],
            [0],
            marker="D",
            color="none",
            markeredgecolor="grey",
            markersize=6,
            linewidth=0,
            label="◇ templateFlux (open)",
        ),
    ]
    axes_flux[0].legend(handles=type_handles, fontsize=7, loc="best")

    # ── Super-title ───────────────────────────────────────────────────────────
    fig.suptitle(
        f"rank #{rank + 1}  |  diaObjectId={obj_id}  |  field={field}  |"
        f"  N={len(df_obj)}  |  t₀={t0_date}  |  Δt_max={global_dt_max:.1f} d\n"
        "Top row : psfFlux □ / scienceFlux ● / templateFlux ◇  [nJy]  "
        "— colour = Δt (blue → red)\n"
        "Bottom row : CCD trajectory on LSSTCam focal plane — same colour coding",
        fontsize=10,
        y=1.02,
    )

    savefig(f"flux_focalplane_{src_label}_rank{rank + 1:02d}_obj{obj_id}")
    plt.show()


print("plot_flux_and_focalplane() defined.")

## 9. Main loop — one figure per top-N DIA object

In [ ]:
if not has_science:
    print(
        "WARNING: scienceFlux column not found — only psfFlux and templateFlux (if available) will be plotted."
    )

for rank, obj_id in enumerate(top_objects):
    df_obj = df_top[df_top[COL_OBJ] == obj_id].sort_values(COL_MJD)
    plot_flux_and_focalplane(obj_id, df_obj, src_label, rank)

## 10. Summary: per-object CCD census per band

In [ ]:
records = []
for rank, obj_id in enumerate(top_objects):
    df_obj = df_top[df_top[COL_OBJ] == obj_id]
    t0_mjd = float(df_obj[COL_MJD].min())
    t0_date = mjd_to_datestr([t0_mjd])[0]
    all_dets = sorted(df_obj[COL_DET].dropna().unique().astype(int).tolist())

    row = {
        "rank": rank + 1,
        "diaObjectId": obj_id,
        "n_alerts": len(df_obj),
        "t0_date": t0_date,
        "dt_max_days": round(float((df_obj[COL_MJD].values.astype(float) - t0_mjd).max()), 1),
        "n_det_total": len(all_dets),
        "det_ids": ",".join(str(d) for d in all_dets),
    }
    for band in BANDS:
        df_b = df_obj[df_obj[COL_BAND] == band]
        if len(df_b) == 0:
            row[f"n_obs_{band}"] = 0
            row[f"n_det_{band}"] = 0
            row[f"det_{band}"] = ""
            continue
        dets_b = sorted(df_b[COL_DET].dropna().unique().astype(int).tolist())
        row[f"n_obs_{band}"] = len(df_b)
        row[f"n_det_{band}"] = len(dets_b)
        row[f"det_{band}"] = ",".join(str(d) for d in dets_b)
    records.append(row)

df_summary = pd.DataFrame(records)
print("CCD census per DIA object and band:")
display(df_summary)

In [ ]:
out_path = os.path.join(DIR_FIGS, f"flux_focalplane_summary_{src_label}.csv")
df_summary.to_csv(out_path, index=False)
print(f"Saved → {out_path}")

## 11. Interpretation guide

| Pattern in the two rows | Interpretation |
|---|---|
| Flux outliers in row 0 correspond to a **specific CCD colour** in row 1 | That CCD has a systematic (flat-field, gain, bad columns) |
| Flux level changes abruptly at a certain **Δt** (colour transition) | Template epoch boundary; atmosphere change; instrument reconfiguration |
| All CCDs in row 1 contribute equally to the scatter | Global calibration issue (zeropoint, transparency) |
| Only **one or two CCDs** in a given band | Sparse sampling — trajectory is well constrained but statistics are poor |
| psfFlux (□) varies but scienceFlux (●) is stable | Template noise is dominant |
| scienceFlux (●) varies independently of CCD position | Source is intrinsically variable |

**Next step:** for the suspect CCD(s) identified here, correlate with visit-level
metadata (seeing, sky, zeropoint) available in `visit_summary_src.csv`.